In [5]:
import kagglehub
import pandas as pd
from pathlib import Path

### Download the dataset

In [6]:
# Download latest version
path = kagglehub.dataset_download("ellipticco/elliptic-data-set")
print("Path to dataset files:", path)

# Path
data_dir = Path(path) / "elliptic_bitcoin_dataset"

Path to dataset files: /Users/aditsingh/.cache/kagglehub/datasets/ellipticco/elliptic-data-set/versions/1


In [ ]:
features_path = data_dir / "elliptic_txs_features.csv"
edges_path    = data_dir / "elliptic_txs_edgelist.csv"
classes_path  = data_dir / "elliptic_txs_classes.csv"

features = pd.read_csv(features_path, header=None)
edges    = pd.read_csv(edges_path)
classes  = pd.read_csv(classes_path)

In [14]:
# Shapes
print("features: ", features.shape)
print("edges: ", edges.shape)
print("classes: ", classes.shape)
print("\n")
print(classes["class"].value_counts())
usable = classes[classes["class"] != "unknown"]
print(usable["class"].value_counts(normalize=True))

features:  (203769, 167)
edges:  (234355, 2)
classes:  (203769, 2)


class
unknown    157205
2           42019
1            4545
Name: count, dtype: int64
class
2    0.902392
1    0.097608
Name: proportion, dtype: float64


### Merge the dataset.

In [ ]:
features.rename(columns={0: 'txId', 1: 'time_step'}, inplace=True)

nodes = features.merge(classes, on ='txId', how='left')

print(nodes.head())

        txId  time_step         2         3         4          5         6  \
0  230425980          1 -0.171469 -0.184668 -1.201369  -0.121970 -0.043875   
1    5530458          1 -0.171484 -0.184668 -1.201369  -0.121970 -0.043875   
2  232022460          1 -0.172107 -0.184668 -1.201369  -0.121970 -0.043875   
3  232438397          1  0.163054  1.963790 -0.646376  12.409294 -0.063725   
4  230460314          1  1.011523 -0.081127 -1.201369   1.153668  0.333276   

          7          8         9  ...       158       159       160       161  \
0 -0.113002  -0.061584 -0.162097  ... -0.600999  1.461330  1.461369  0.018279   
1 -0.113002  -0.061584 -0.162112  ...  0.673103 -0.979074 -0.978556  0.018279   
2 -0.113002  -0.061584 -0.162749  ...  0.439728 -0.979074 -0.978556 -0.098889   
3  9.782742  12.414558 -0.163645  ... -0.613614  0.241128  0.241406  1.072793   
4  1.312656  -0.061584 -0.163523  ... -0.400422  0.517257  0.579382  0.018279   

        162       163       164       165   

### Verify unique timestamps

In [22]:
nodes["time_step"].nunique()
nodes["time_step"].value_counts().sort_index()

time_step
1     7880
2     4544
3     6621
4     5693
5     6803
6     4328
7     6048
8     4457
9     4996
10    6727
11    4296
12    2047
13    4528
14    2022
15    3639
16    2975
17    3385
18    1976
19    3506
20    4291
21    3537
22    5894
23    4165
24    4592
25    2314
26    2523
27    1089
28    1653
29    4275
30    2483
31    2816
32    4525
33    3151
34    2486
35    5507
36    6393
37    3306
38    2891
39    2760
40    4481
41    5342
42    7140
43    5063
44    4975
45    5598
46    3519
47    5121
48    2954
49    2454
Name: count, dtype: int64

### Verify that no edges cross timesteps. This should not occur in the dataset and will need to be handled accordingly, as it would mean that a transaction spans across two different timesteps. 

In [26]:
edges_time_table = (
        edges.merge(nodes[["txId", "time_step"]], left_on="txId1", right_on="txId")
        .rename(columns={"time_step" : "t1"})
        .merge(nodes[["txId", "time_step"]], left_on="txId2", right_on="txId")
        .rename(columns={"time_step": "t2"})
    )

print((edges_time_table["t1"] != edges_time_table["t2"]).sum())



0
